In [2]:
import json

path = r"D:\GitHub\ChatBot\output_nghidinh\chunks_clean.json"
data = json.load(open(path, "r", encoding="utf-8"))
if isinstance(data, dict) and "chunks" in data:
    data = data["chunks"]

kw = ["Bộ Tư pháp", "tư pháp", "TƯ PHÁP"]
cnt = 0
for c in data:
    t = c.get("text","")
    if any(k in t for k in kw):
        cnt += 1
print("Chunks containing tư pháp:", cnt, "/", len(data))

Chunks containing tư pháp: 17 / 1861


In [3]:
import json
from collections import Counter

path = r"D:\GitHub\ChatBot\output_nghidinh\chunks_clean.json"
data = json.load(open(path, "r", encoding="utf-8"))
if isinstance(data, dict) and "chunks" in data:
    data = data["chunks"]

vb = [c.get("metadata",{}).get("van_ban","") for c in data]
ctr = Counter(vb)
for name, n in ctr.most_common(10):
    print(n, "|", name[:120])
print("Unique van_ban:", len(ctr))

187 | NGHỊ ĐỊNH Quy định phân định thẩm quyền của chính quyền địa phương 02 cấp trong lĩnh vực quản lý nhà nước của Bộ Nông ng
170 | NGHỊ ĐỊNH Quy định về phân quyền, phân cấp trong lĩnh vực Gi3: ĐẾN quản lý nhà nước của Bộ Khoa học và Công nghệ Ngày 45
170 | NGHỊ ĐỊNH Quy định về phân định thẩm quyền của chính quyền địa phương cổng thông tin điện tử chính phủ 192 cấp trong lĩn
165 | NGHỊ ĐỊNH "Quy định về phân định thẩm quyền của chính quyền địa phương 02 cấp trong lĩnh vực quản lý nhà nước của Bộ Nội
150 | NGHỊ ĐỊNH Quy định về phân định thẩm quyền của chính quyền địa phương 02 cấp trong lĩnh vực quản lý nhà nước của bộ tài 
135 | NGHỊ ĐỊNH Quy định tổ chức các cơ quan chuyên môn thuộc Ủy ban nhân dân tỉnh, thành phố trực thuộc trung ương và Ủy ban 
111 | NGHỊ ĐỊNH Quy định về phân định thẩm quyền của chính quyền địa phương 02 cấp trong lĩnh vực quản lý nhà nước của Bộ Y tế
90 | NGHỊ ĐỊNH Quy định về phân quyền, phân cấp trong quản lý nhà nước lĩnh vực nội vụ
84 | NGHỊ ĐỊNH Quy định 

In [ ]:
import json
from collections import Counter

path = r"D:\GitHub\ChatBot\output_nghidinh\chunks_clean.json"
data = json.load(open(path, "r", encoding="utf-8"))
if isinstance(data, dict) and "chunks" in data:
    data = data["chunks"]

vb = [c.get("metadata",{}).get("van_ban","") for c in data]
ctr = Counter(vb)
for name, n in ctr.most_common(10):
    print(n, "|", name[:120])
print("Unique van_ban:", len(ctr))

187 | NGHỊ ĐỊNH Quy định phân định thẩm quyền của chính quyền địa phương 02 cấp trong lĩnh vực quản lý nhà nước của Bộ Nông ng
170 | NGHỊ ĐỊNH Quy định về phân quyền, phân cấp trong lĩnh vực Gi3: ĐẾN quản lý nhà nước của Bộ Khoa học và Công nghệ Ngày 45
170 | NGHỊ ĐỊNH Quy định về phân định thẩm quyền của chính quyền địa phương cổng thông tin điện tử chính phủ 192 cấp trong lĩn
165 | NGHỊ ĐỊNH "Quy định về phân định thẩm quyền của chính quyền địa phương 02 cấp trong lĩnh vực quản lý nhà nước của Bộ Nội
150 | NGHỊ ĐỊNH Quy định về phân định thẩm quyền của chính quyền địa phương 02 cấp trong lĩnh vực quản lý nhà nước của bộ tài 
135 | NGHỊ ĐỊNH Quy định tổ chức các cơ quan chuyên môn thuộc Ủy ban nhân dân tỉnh, thành phố trực thuộc trung ương và Ủy ban 
111 | NGHỊ ĐỊNH Quy định về phân định thẩm quyền của chính quyền địa phương 02 cấp trong lĩnh vực quản lý nhà nước của Bộ Y tế
90 | NGHỊ ĐỊNH Quy định về phân quyền, phân cấp trong quản lý nhà nước lĩnh vực nội vụ
84 | NGHỊ ĐỊNH Quy định 

In [9]:
import json
import numpy as np
import faiss
from sentence_transformers import SentenceTransformer

def load_chunks(chunks_path: str):
    with open(chunks_path, "r", encoding="utf-8") as f:
        chunks = json.load(f)
    if isinstance(chunks, dict) and "chunks" in chunks:
        chunks = chunks["chunks"]
    return chunks

def retrieve_notebook(
    question: str,
    topk: int = 5,
    topn: int = 300,
    filter_kw: str = "",
    chunks_path: str = r"D:\GitHub\ChatBot\output_nghidinh\chunks_clean.json",
    index_path: str = r"D:\GitHub\ChatBot\vector_data\legal_hf_cosine\index.faiss",
    model_name: str = "Quockhanh05/Vietnam_legal_embeddings",
    device: str = "cuda",
):
    # Load
    chunks = load_chunks(chunks_path)
    index = faiss.read_index(index_path)
    encoder = SentenceTransformer(model_name, device=device)

    # Encode query (normalize -> IP = cosine)
    q_emb = encoder.encode([question], normalize_embeddings=True)
    q_emb = np.asarray(q_emb, dtype="float32")

    # Search rộng
    scores, ids = index.search(q_emb, topn)
    scores = scores[0].tolist()
    ids = ids[0].tolist()

    kw = filter_kw.strip().lower()

    # Filter candidates
    candidates = []
    for idx, score in zip(ids, scores):
        if idx < 0 or idx >= len(chunks):
            continue
        ch = chunks[idx]
        meta = ch.get("metadata", {})
        vb = str(meta.get("van_ban", "")).lower()
        tx = str(ch.get("text", "")).lower()

        if kw and (kw not in vb) and (kw not in tx):
            continue

        candidates.append((idx, float(score)))

    # Sort + take topk after filtering
    candidates = sorted(candidates, key=lambda x: x[1], reverse=True)[:topk]

    # Build results from candidates (✅ đúng)
    results = []
    for rank, (idx, score) in enumerate(candidates, start=1):
        ch = chunks[idx]
        meta = ch.get("metadata", {})
        results.append({
            "rank": rank,
            "score": score,
            "text": ch.get("text", ""),
            "metadata": {
                "van_ban": meta.get("van_ban", ""),
                "chuong": meta.get("chuong", ""),
                "dieu": meta.get("dieu", ""),
                "khoan": meta.get("khoan", ""),
                "nguon": meta.get("nguon", ""),
                "chunk_id": idx
            }
        })

    # Pretty print
    print("=" * 90)
    print("QUESTION:", question)
    print(f"topn={topn}, filter_kw='{filter_kw}', topk={topk} | kept={len(candidates)}")
    print("=" * 90)
    for r in results:
        md = r["metadata"]
        print(f"[{r['rank']}] score={r['score']:.4f} | {md['van_ban']} | Chương {md['chuong']} | Điều {md['dieu']} | Khoản {md['khoan']} | chunk_id={md['chunk_id']}")
        preview = r["text"].replace("\n", " ")
        print("    ", preview[:260] + ("..." if len(preview) > 260 else ""))
        print("-" * 90)

    return {
        "question": question,
        "topk": topk,
        "topn": topn,
        "filter_kw": filter_kw,
        "results": results
    }

In [10]:
out = retrieve_notebook(
    "Thẩm quyền của cấp xã trong lĩnh vực tư pháp là gì?",
    topk=5,
    topn=400,
    filter_kw="tư pháp",
    device="cuda"
)

QUESTION: Thẩm quyền của cấp xã trong lĩnh vực tư pháp là gì?
topn=400, filter_kw='tư pháp', topk=5 | kept=5
[1] score=0.4293 | NGHỊ ĐỊNH Quy định về phân định thẩm quyền của chính quyền địa phương 02 cấp trong lĩnh vực quản lý nhà nước của Bộ Tư pháp | Chương II | Điều 4 | Khoản None | chunk_id=11
     Theo Điều 4 Thẩm quyền đăng ký hộ tịch Ủy ban nhân dân xã, phường, đặc khu (sau đây gọi là Ủy ban Nhân dân cấp xã) thực hiện thẩm quyền đăng ký hộ tịch quy định tại khoản 2 Điều 7, Chương III của Luật hộ tịch năm 2014 (sau đây gọi là Luật Hộ tịch), các Điều 1,...
------------------------------------------------------------------------------------------
[2] score=0.4226 | NGHỊ ĐỊNH Quy định về phân định thẩm quyền của chính quyền địa phương 02 cấp trong lĩnh vực quản lý nhà nước của Bộ Tư pháp | Chương I | Điều 2 | Khoản 4 | chunk_id=4
     Khoản 4 Điều 2 Xác định rõ nội dung và phạm vi nhiệm vụ, quyền hạn mà chính quyền địa phương được quyết định, tổ chức thực hiện và chịu trách nhiệm v

In [1]:
import json

CHUNKS_PATH = r"D:\GitHub\ChatBot\output_nghidinh\chunks_clean_norm.json"
chunks = json.load(open(CHUNKS_PATH, "r", encoding="utf-8"))
if isinstance(chunks, dict) and "chunks" in chunks:
    chunks = chunks["chunks"]

def count_vanban(substr):
    s = substr.lower()
    return sum(s in c.get("metadata",{}).get("van_ban","").lower() for c in chunks)

def count_text(substr):
    s = substr.lower()
    return sum(s in c.get("text","").lower() for c in chunks)

print("Total:", len(chunks))
print("Bộ Tư pháp:", count_vanban("bộ tư pháp"))
print("Bộ Tài chính:", count_vanban("bộ tài chính"))
print("Bộ Y tế:", count_vanban("bộ y tế"))
print("Contains 'lệ phí trước bạ' in text:", count_text("lệ phí trước bạ"))
print("Contains 'trước bạ' in text:", count_text("trước bạ"))

Total: 1861
Bộ Tư pháp: 136
Bộ Tài chính: 150
Bộ Y tế: 111
Contains 'lệ phí trước bạ' in text: 2
Contains 'trước bạ' in text: 2
